In [1]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from downloading_the_base_model.download_model import downlaod_model

downlaod_model("Qwen/Qwen3-0.6B", "qwen")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

'/teamspace/studios/this_studio/open-posttraining-system/evaluating_reasoning_models/qwen'

In [2]:


import torch
from safetensors.torch import load_file

from base_model.qwen import (
    QWEN_CONFIG_06_B,
    Qwen3Model,
    Qwen3Tokenizer,
    generate_text,
    load_hf_weights_into_qwen,
)

model_dir = Path.cwd() / "qwen"

#def main(prompt):
def load_model_and_tokenizer(which_model, use_compile):

    if which_model == "base":
        tokenizer = Qwen3Tokenizer(model_dir / "tokenizer.json")
        
    elif which_model == "reasoning":
        tokenizer = Qwen3Tokenizer(model_dir / "tokenizer.json", apply_chat_template=True,   add_generation_prompt=True, add_thinking=True)
    
    

    else:
        raise ValueError(f"Not a valid model type")
    
    weights = load_file(model_dir / "model.safetensors")

    model = Qwen3Model(QWEN_CONFIG_06_B)
    load_hf_weights_into_qwen(
        model,
        param_config={
            "n_layers": QWEN_CONFIG_06_B["n_layers"],
            "hidden_dim": QWEN_CONFIG_06_B["hidden_dim"],
        },
        params=weights,
    )
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.to(torch.bfloat16)

    if use_compile:
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        model = torch.compile(model)
     
    return model, tokenizer




In [3]:
### here
which_model = "base"


model, tokenizer = load_model_and_tokenizer(
    which_model=which_model,
    use_compile=False
)



In [4]:
import torch

from base_model.qwen import KVCache


device = "cuda" if torch.cuda.is_available() else "cpu"


@torch.inference_mode()
def generate_text_stream_with_kv_cache(prompt, model, tokenizer, device, max_new_tokens, eos_token_id):
    # Encode prompt and move to device
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)

    # Set model to evaluation mode
    model.eval()

    # Initialize KV cache
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # Initial forward pass using full prompt
    logits = model(input_ids, cache=cache)[:, -1]

    # Autoregressive generation loop
    for _ in range(max_new_tokens):

        # Greedy decoding
        next_token = torch.argmax(logits,dim=-1,keepdim=True)

        # Stop if EOS token is generated
        if (eos_token_id is not None and torch.all(next_token == eos_token_id)):
            break

        # Stream generated token
        yield next_token

        # Next forward pass only uses new token
        logits = model(next_token, cache=cache)[:, -1]

def generate_text(prompt, model, tokenizer, device, max_new_tokens, eos_token_id):


    for token in generate_text_stream_with_kv_cache(
        prompt=prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        eos_token_id=eos_token_id,
        device=device,
    ):
        output_tokens = token.squeeze(0).tolist()

        print(tokenizer.decode(output_tokens), end="", flush=True)

    
         

    


In [5]:
max_new_tokens = 600
device = "cuda" if torch.cuda.is_available() else "cpu"
eos_token_id = tokenizer.eos_token_id
prompt = ("If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?")

generate_text(prompt, model, tokenizer, device, max_new_tokens, eos_token_id)

 To

 solve this problem, we can use the identity that relates the sum and product of two numbers to their squares. The identity is:

$$
a^2 + b^2 = (a + b)^2 - 2ab
$$

Given that $a + b = 3$ and $ab = \frac{13}{6}$, we can substitute these values into the identity to find $a^2 + b^2$.

Let's compute this step by step:

1. Start with the given values:
   $$
   a + b = 3
   $$
   $$
   ab = \frac{13}{6}
   $$

2. Apply the identity:
   $$
   a^2 + b^2 = (a + b)^2 - 2ab
   $$

3. Substitute the given values:
   $$
   a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
   $$

4. Simplify the expression:
   $$
   a^2 + b^2 = 9 - \frac{26}{6}
   $$

5. Simplify the fraction:
   $$
   a^2 + b^2 = 9 - \frac{13}{3}
   $$

6. Convert 9 to a fraction with denominator 3:
   $$
   a^2 + b^2 = \frac{27}{3} - \frac{13}{3}
   $$

7. Subtract the fractions:
   $$
   a^2 + b^2 = \frac{14}{3}
   $$

So, the value of $a^2 + b^2$ is $\frac{14}{3}$.

Let me double-check the calculations to ensure there are no err

In [7]:
from IPython.display import Math

display(Math(r"\dfrac{14}{3}"))

<IPython.core.display.Math object>

### creating a wrapper function

In [8]:
## ok writing the kv_cache token_stream generation and text_generation seperately
from base_model.qwen import KVCache
@torch.inference_mode()
def kvcache_based_token_generation(input_ids, model, max_new_tokens, eos_token_id):
    # initializing model in eval_model() 
    model.eval()   
    cache = KVCache(n_layers = model.cfg["n_layers"]) 
    model.reset_kv_cache()

    logits = model(input_ids, cache=cache)[:,-1]

    for _ in range(max_new_tokens):
        next_token = torch.argmax(logits, dim=-1, keepdim=True)

        if (eos_token_id is not None and 
            torch.all(next_token == eos_token_id)):
            break

        yield next_token

        logits = model(next_token, cache=cache)[:, -1]






## this function will use the existing kv_cache stream function---
def text_generation_wrapper(prompt, model, tokenizer, device, max_new_tokens, verbose=False):
    ## encode 
    input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)


    generated_ids = []
    for token in kvcache_based_token_generation(input_ids, model, max_new_tokens, tokenizer.eos_token_id):
        output_tokens = token.squeeze(0).tolist()
        ## append ids --scalar
        generated_ids.append(token.squeeze(0).item())
        if verbose:
            print(tokenizer.decode(output_tokens), end="", flush=True)

    return tokenizer.decode(generated_ids)


skip_portion = False

if not skip_portion:
    generated_text = text_generation_wrapper(prompt, model, tokenizer, device, max_new_tokens=600, verbose=True)
    




 To solve this problem, we can use the identity that relates the sum and product of

 two numbers to their squares. The identity is:

$$
a^2 + b^2 = (a + b)^2 - 2ab
$$

Given that $a + b = 3$ and $ab = \frac{13}{6}$, we can substitute these values into the identity to find $a^2 + b^2$.

Let's compute this step by step:

1. Start with the given values:
   $$
   a + b = 3
   $$
   $$
   ab = \frac{13}{6}
   $$

2. Apply the identity:
   $$
   a^2 + b^2 = (a + b)^2 - 2ab
   $$

3. Substitute the given values:
   $$
   a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
   $$

4. Simplify the expression:
   $$
   a^2 + b^2 = 9 - \frac{26}{6}
   $$

5. Simplify the fraction:
   $$
   a^2 + b^2 = 9 - \frac{13}{3}
   $$

6. Convert 9 to a fraction with denominator 3:
   $$
   a^2 + b^2 = \frac{27}{3} - \frac{13}{3}
   $$

7. Subtract the fractions:
   $$
   a^2 + b^2 = \frac{14}{3}
   $$

So, the value of $a^2 + b^2$ is $\frac{14}{3}$.

Let me double-check the calculations to ensure there are no errors:

- $ (3)^2 = 9 $
- $ 2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3} $


In [9]:
from IPython.display import Latex, display
display(Latex(generated_text))

<IPython.core.display.Latex object>

In [10]:
generated_text

" To solve this problem, we can use the identity that relates the sum and product of two numbers to their squares. The identity is:\n\n$$\na^2 + b^2 = (a + b)^2 - 2ab\n$$\n\nGiven that $a + b = 3$ and $ab = \\frac{13}{6}$, we can substitute these values into the identity to find $a^2 + b^2$.\n\nLet's compute this step by step:\n\n1. Start with the given values:\n   $$\n   a + b = 3\n   $$\n   $$\n   ab = \\frac{13}{6}\n   $$\n\n2. Apply the identity:\n   $$\n   a^2 + b^2 = (a + b)^2 - 2ab\n   $$\n\n3. Substitute the given values:\n   $$\n   a^2 + b^2 = (3)^2 - 2 \\left( \\frac{13}{6} \\right)\n   $$\n\n4. Simplify the expression:\n   $$\n   a^2 + b^2 = 9 - \\frac{26}{6}\n   $$\n\n5. Simplify the fraction:\n   $$\n   a^2 + b^2 = 9 - \\frac{13}{3}\n   $$\n\n6. Convert 9 to a fraction with denominator 3:\n   $$\n   a^2 + b^2 = \\frac{27}{3} - \\frac{13}{3}\n   $$\n\n7. Subtract the fractions:\n   $$\n   a^2 + b^2 = \\frac{14}{3}\n   $$\n\nSo, the value of $a^2 + b^2$ is $\\frac{14}{3}$.\n